ACTIVIDAD SESIÓN ENTRENAMIENTO Y OPTIMIZACIÓN DE REDES NEURONALES
Eres un analista de datos en una agencia espacial encargada de estudiar planetas de distintos
sistemas solares. Tu misión es construir una red neuronal que prediga si un planeta es habitable o
no, basándose en sus características físicas y atmosféricas.
Para esto, utilizarás datos de exoplanetas consumidos desde la API pública de la NASA:


https://exoplanetarchive.ipac.caltech.edu/docs/program_interfaces.html
El modelo analizará información de distintos planetas y determinará la probabilidad de que sean
habitables, basándose en su tamaño, temperatura y distancia a su estrella.


In [ ]:
import requests
import pandas as pd
from io import StringIO
 
# URL de la API
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
 
# Consulta SQL con los 5 campos solicitados
query = """
SELECT pl_name, pl_rade, pl_bmasse, sy_dist, pl_eqt, st_spectype
FROM ps
WHERE pl_rade IS NOT NULL
AND pl_bmasse IS NOT NULL
AND sy_dist IS NOT NULL
AND pl_eqt IS NOT NULL
AND st_spectype IS NOT NULL
"""
 
# Ejecutar la consulta
params = {
    "query": query,
    "format": "csv"
}
 
response = requests.get(url, params=params)
df = pd.read_csv(StringIO(response.text))
 
print(df.head())


        pl_name  pl_rade  pl_bmasse    sy_dist  pl_eqt st_spectype
0    HAT-P-11 b    4.730     25.743    37.7647   878.0          K4
1   Kepler-22 b    2.380     36.000   194.6420   262.0        G5 V
2  Kepler-117 c   12.341    584.780  1455.5700   704.0        F8 V
3  Kepler-117 b    8.059     29.875  1455.5700   984.0        F8 V
4  Kepler-425 b   10.962     79.450   646.6470  1070.0        K1 V


In [2]:
# Actividad del módulo 8 sesión 4
# Nota didáctica: El archivo de la NASA no trae una etiqueta “habitable”. Creamos una etiqueta binaria (0/1) con un criterio simple y
# explicable basado en literatura popular: radio ~[0.5, 1.75], masa ~[0.1, 5], temperatura de equilibrio ~[240, 320] K, estrella
# tipo F/G/K/M y órbita razonable. Esto se puede ajustar y debatir con la clase.
# ================================================================
# SESIÓN: Entrenamiento y Optimización – Clasificación de Habitabilidad
# Fuente de datos: NASA Exoplanet Archive (API pública)
# Docs: https://exoplanetarchive.ipac.caltech.edu/docs/program_interfaces.html
# ================================================================
 
# ===== 0) IMPORTS Y CONFIGURACIÓN =====
import os, io, sys, math, textwrap
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score)
 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
 
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
 
print("TensorFlow:", tf.__version__)# ===== 1) CONSUMO Y PREPROCESAMIENTO =====
# 1.1 Descarga desde la API de la NASA (TAP, formato CSV).
# Tomamos de la tabla "pscomppars" columnas necesarias:
# - pl_name (nombre)
# - pl_rade (radio en radios terrestres)
# - pl_bmasse (masa en masas terrestres)
# - pl_orbsmax (semi-eje mayor en AU, aproximación de distancia a la estrella)
# - pl_eqt (temperatura de equilibrio en Kelvin)
# - st_spectype (tipo espectral de la estrella; a veces viene como 'G2V', etc.)
 
TAP_URL = (
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?"
    "query="
    "select+pl_name,pl_rade,pl_bmasse,pl_orbsmax,pl_eqt,st_spectype+"
    "from+pscomppars&format=csv"
)
 
def fetch_exoplanets_csv():
    try:
        r = requests.get(TAP_URL, timeout=60)
        r.raise_for_status()
        return pd.read_csv(io.StringIO(r.text))
    except Exception as e:
        print("⚠️ No se pudo descargar desde la API (usando respaldo pequeño). Error:", e)
        # Mini respaldo en caso de no tener internet (5 filas de ejemplo, edítalo si deseas)
        csv_fallback = """pl_name,pl_rade,pl_bmasse,pl_orbsmax,pl_eqt,st_spectype
Kepler-452 b,1.6,5.0,1.05,265,G2V
Kepler-442 b,1.34,2.3,0.41,233,K5
TRAPPIST-1 e,0.92,0.77,0.029,251,M8
Kepler-62 f,1.41,2.8,0.718,270,K2
K2-18 b,2.6,8.6,0.159,265,M2
"""
        return pd.read_csv(io.StringIO(csv_fallback))
 
df_raw = fetch_exoplanets_csv()
print("Filas/Columnas crudo:", df_raw.shape)
df_raw.head()

ModuleNotFoundError: No module named 'tensorflow'


INSTRUCCIONES
1. Consumo y preprocesamiento de datos (3 puntos)
1. Conectar con la API de la NASA y extraer un conjunto de datos relevante.
2. Seleccionar las siguientes características para el modelo:
o Radio del planeta
o Masa del planeta
o Distancia a su estrella
o Temperatura de equilibrio
o Tipo de estrella anfitriona
3. Limpiar los datos eliminando valores nulos o incorrectos.
2. Creación de la red neuronal (2 puntos)
1. Crear una red neuronal con tres capas ocultas y una capa de salida con activación sigmoide.
2. Utilizar funciones de activación ReLU en las capas ocultas.
3. Aplicar una técnica de regularización (como dropout o L2).